In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("C:\\Users\\rahul\\OneDrive\\Desktop\\Rahul Job\\Fraud Transactions Analysis Project\\Fraud_Transaction_Analysis_Rawdataset.csv")

In [5]:
df.head(10)

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,3094,200,1.0,POS,Food,US,9,0.106584,0.180465,0
1,6904,310,1.0,POS,Food,DE,14,0.110509,0.202540,0
2,4087,216,1.0,Online,Grocery,DE,7,0.236457,0.019812,0
3,3199,846,1.0,ATM,Electronics,US,21,0.050318,0.078093,0
4,4538,339,1.0,ATM,Clothing,TR,10,0.214854,0.005979,0
5,434,433,1.0,Online,Grocery,TR,20,0.205439,0.035571,0
6,1773,919,1.0,ATM,Food,DE,8,0.018977,0.185775,0
7,5569,423,1.0,ATM,Travel,UK,6,0.123899,0.080673,0
8,7931,665,1.0,POS,Clothing,UK,19,0.038019,0.172416,0
9,2071,974,1.0,ATM,Clothing,FR,12,0.133466,0.028577,0


In [32]:
# Data Exploration
df.shape

(10000, 12)

In [8]:
df.dtypes

transaction_id         int64
user_id                int64
amount               float64
transaction_type      object
merchant_category     object
country               object
hour                   int64
device_risk_score    float64
ip_risk_score        float64
is_fraud               int64
dtype: object

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     10000 non-null  int64  
 1   user_id            10000 non-null  int64  
 2   amount             10000 non-null  float64
 3   transaction_type   10000 non-null  object 
 4   merchant_category  10000 non-null  object 
 5   country            10000 non-null  object 
 6   hour               10000 non-null  int64  
 7   device_risk_score  10000 non-null  float64
 8   ip_risk_score      10000 non-null  float64
 9   is_fraud           10000 non-null  int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 781.4+ KB


In [11]:
df.isnull().sum()

transaction_id       0
user_id              0
amount               0
transaction_type     0
merchant_category    0
country              0
hour                 0
device_risk_score    0
ip_risk_score        0
is_fraud             0
dtype: int64

In [33]:
# Data Cleaning
df['transaction_type'] = df['transaction_type'].astype('category')

In [13]:
df['merchant_category'] = df['merchant_category'].astype('category')
df['country'] = df['country'].astype('category')

In [14]:
df.dtypes

transaction_id          int64
user_id                 int64
amount                float64
transaction_type     category
merchant_category    category
country              category
hour                    int64
device_risk_score     float64
ip_risk_score         float64
is_fraud                int64
dtype: object

In [15]:
df.describe()

,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,is_fraud
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4999.50000,500.058700,178.142763,14.247100,0.183773,0.184669,0.050000
std,2886.89568,288.328495,531.647950,5.347383,0.177381,0.175772,0.217956
min,0.00000,0.000000,1.000000,0.000000,0.000030,0.000009,0.000000
25%,2499.75000,247.000000,65.084753,10.000000,0.075721,0.077762,0.000000
50%,4999.50000,503.000000,101.686510,14.000000,0.156583,0.158290,0.000000
75%,7499.25000,750.250000,138.280872,19.000000,0.234939,0.236968,0.000000
max,9999.00000,999.000000,11628.213880,23.000000,0.998737,0.999603,1.000000


In [30]:
print(df.duplicated().sum())

0


In [41]:
# Feature Engeneering
df['is_high_amount'] = (df['amount'] > 1000).astype(int)
df['is_night_txn'] = df['hour'].apply(lambda h: 1 if h < 6 or h>=22 else 0)
df['is_high_risk'] = ((df['device_risk_score'] > 0.7) & (df['ip_risk_score'] > 0.7)).astype(int)
df['amount_bucket'] = pd.cut(df['amount'], bins=[0,100,500,1000,5000,float('inf')], labels =['<100','100-500','500-1000','1000-5000','5000+'])

In [42]:
df.head()

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud,is_high_amount,is_night_txn,is_high_risk,amount_bucket
0,3094,200,1.0,POS,Food,US,9,0.106584,0.180465,0,0,0,0,<100
1,6904,310,1.0,POS,Food,DE,14,0.110509,0.202540,0,0,0,0,<100
2,4087,216,1.0,Online,Grocery,DE,7,0.236457,0.019812,0,0,0,0,<100
3,3199,846,1.0,ATM,Electronics,US,21,0.050318,0.078093,0,0,0,0,<100
4,4538,339,1.0,ATM,Clothing,TR,10,0.214854,0.005979,0,0,0,0,<100


In [43]:
pip install sqlalchemy pymysql pandas

Note: you may need to restart the kernel to use updated packages.


In [49]:
from sqlalchemy import create_engine, text

USERNAME = 'root'
PASSWORD = 'Rahul07'
HOST     = 'localhost'
PORT     = '3306'
DATABASE = 'fraud_db'

base_engine = create_engine(
    f'mysql+pymysql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}'
)

with base_engine.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {DATABASE}'))
    print(f'✅ Database `{DATABASE}` is ready')

✅ Database `fraud_db` is ready


In [52]:
engine = create_engine(f'mysql+pymysql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}')
df.to_sql(name='fraud_transactions', con=engine, if_exists='replace', index=False, chunksize=500)

10000

In [54]:
pd.read_sql('SELECT * FROM fraud_transactions LIMIT 5', engine)

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud,is_high_amount,is_night_txn,is_high_risk,amount_bucket
0,3094,200,1.0,POS,Food,US,9,0.106584,0.180465,0,0,0,0,<100
1,6904,310,1.0,POS,Food,DE,14,0.110509,0.202540,0,0,0,0,<100
2,4087,216,1.0,Online,Grocery,DE,7,0.236457,0.019812,0,0,0,0,<100
3,3199,846,1.0,ATM,Electronics,US,21,0.050318,0.078093,0,0,0,0,<100
4,4538,339,1.0,ATM,Clothing,TR,10,0.214854,0.005979,0,0,0,0,<100
